# TP 21 — Linear probing : ViT pré-entraîné sur STL-10

**Objectifs**

- Réutiliser un Vision Transformer pré-entraîné comme **extracteur de features**, sans le ré-entraîner.
- Entraîner uniquement une **tête légère** (logistic regression, k-NN) sur les features.
- Comparer à un k-NN sur features L2-normalisées.
- Visualiser la séparabilité des classes en 2D (t-SNE).

**Durée indicative : 60 minutes.**

**Modèle :** `vit_small_patch16_224.augreg_in21k_ft_in1k` (ViT-S/16, ~22M paramètres).
**Dataset :** STL-10 (10 classes, images 96x96, sous-ensemble équilibré).

C'est exactement le paradigme **backbone gelé + tête légère** : on n'entraîne plus le backbone, on extrait des features une fois pour toutes, et on entraîne un classifieur classique par-dessus. Même esprit que le TP HOG + SVM du jour 3, avec un backbone bien plus puissant.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import timm
import torch
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.neighbors import KNeighborsClassifier
from torchvision import transforms
from torchvision.datasets import STL10

## Exercice 1 — Charger STL-10 (sous-ensemble) et le ViT

1. Téléchargez STL-10 train + test via `torchvision.datasets.STL10(root="./data", split="train", download=True)`.
2. Pour gagner du temps sur CPU, prenez un **sous-ensemble équilibré** : 80 images par classe en train, 40 en test. Total : 800 train, 400 test.
3. Récupérez les images sous forme `numpy uint8` `(N, H, W, 3)` et les labels `(N,)`.
4. Chargez le ViT pré-entraîné avec `num_classes=0` (on retire la tête de classif).
5. Affichez 10 images, une par classe, pour vérifier le bon découpage.

<details>
<summary>💡 Coup de pouce — sous-ensemble équilibré et chargement modèle</summary>

**🎯 But :** réduire le coût de l'extraction (forward sur CPU) tout en gardant un dataset équilibré.

**Charger STL-10**

```python
DATA_DIR = Path("./data")
DATA_DIR.mkdir(exist_ok=True)
ds_train = STL10(root=str(DATA_DIR), split="train", download=True)
ds_test  = STL10(root=str(DATA_DIR), split="test",  download=True)
class_names = ds_train.classes                              # 10 noms
```

`STL10` retourne des `PIL.Image`. Conversion en numpy :

```python
def to_numpy(ds):
    X = np.stack([np.array(img) for img, _ in ds])         # (N, 96, 96, 3)
    y = np.array([lbl for _, lbl in ds])
    return X, y
X_train_full, y_train_full = to_numpy(ds_train)             # 5000
X_test_full,  y_test_full  = to_numpy(ds_test)              # 8000
```

**Sous-ensemble équilibré par classe**

```python
def balanced_subset(X, y, k_per_class, seed=0):
    rng = np.random.default_rng(seed)
    idx_keep = []
    for c in range(10):
        idx_c = np.where(y == c)[0]
        idx_keep.append(rng.choice(idx_c, size=k_per_class, replace=False))
    idx = np.concatenate(idx_keep)
    rng.shuffle(idx)
    return X[idx], y[idx]

X_train, y_train = balanced_subset(X_train_full, y_train_full, k_per_class=80)
X_test,  y_test  = balanced_subset(X_test_full,  y_test_full,  k_per_class=40)
print(X_train.shape, X_test.shape)                          # (800,...) (400,...)
```

⚠️ **Pourquoi équilibré ?** Sur STL-10 brut le test set est gigantesque (8000). Sur CPU, l'extraction prendrait trop longtemps. Un sous-ensemble équilibré reste représentatif pour comparer features.

**Charger le ViT**

```python
model = timm.create_model(
    "vit_small_patch16_224.augreg_in21k_ft_in1k",
    pretrained=True,
    num_classes=0,                                          # retire la tête
)
model.eval()
```

**Vérification visuelle**

```python
fig, axes = plt.subplots(2, 5, figsize=(10, 4.5))
for c in range(10):
    idx = np.where(y_train == c)[0][0]
    ax = axes[c // 5, c % 5]
    ax.imshow(X_train[idx]); ax.set_title(class_names[c]); ax.axis("off")
plt.tight_layout(); plt.show()
```

</details>


In [ ]:
# TODO

## Exercice 2 — Extraire les features (token CLS) + mettre en cache

Pour chaque image :
1. Convertir le tableau numpy `(96, 96, 3)` en `PIL.Image`.
2. Preprocesser (Resize → CenterCrop → ToTensor → Normalize ImageNet).
3. Forward sous `torch.no_grad()`, récupérer le token `[CLS]` (= sortie poolée du backbone). Avec `num_classes=0`, `model(x)` renvoie déjà le vecteur `[CLS]` de shape `(B, 384)`.
4. Stocker dans `feats` de shape `(N, 384)`.
5. **Mettez en cache sur disque** (`np.save`) pour ne pas refaire le forward si vous relancez le notebook.

<details>
<summary>💡 Coup de pouce — boucle d'extraction + cache disque</summary>

**🎯 But :** ne faire le forward qu'**une fois**. C'est ce qui rend la suite (linear probe, k-NN, t-SNE) instantanée.

**Preprocessing**

```python
preprocess = transforms.Compose([
    transforms.Resize(256),                                 # upscale 96 -> 256
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
```

⚠️ **Resize avant CenterCrop** : le ViT a été entraîné sur 224x224. STL-10 est en 96x96 → upscale.

**Fonction d'extraction**

```python
def extract_features(model, X):
    feats = np.zeros((len(X), model.embed_dim), dtype=np.float32)
    with torch.no_grad():
        for i, img in enumerate(X):
            pil = Image.fromarray(img)
            x = preprocess(pil).unsqueeze(0)               # (1, 3, 224, 224)
            feats[i] = model(x)[0].numpy()                 # (384,)
            if (i + 1) % 100 == 0:
                print(f"  {i + 1}/{len(X)}")
    return feats
```

⚠️ Sur CPU, comptez ~3-5 minutes pour 800 images.

**Cache disque**

```python
CACHE = Path("./cache")
CACHE.mkdir(exist_ok=True)

def load_or_extract(model, name, X, split):
    p = CACHE / f"feats_{name}_{split}.npy"
    if p.exists():
        print(f"  cache hit : {p}")
        return np.load(p)
    feats = extract_features(model, X)
    np.save(p, feats)
    return feats

feats_tr = load_or_extract(model, "vit_sup", X_train, "train")
feats_te = load_or_extract(model, "vit_sup", X_test,  "test")
print(feats_tr.shape, feats_te.shape)                      # (800, 384) (400, 384)
```

</details>


In [ ]:
# TODO

## Exercice 3 — Linear probe (régression logistique) + k-NN

Évaluer la qualité des features ViT par deux têtes légères :
1. **L2-normalisez** les features (`feats / ||feats||_2`).
2. Entraînez une `LogisticRegression(max_iter=1000)` sur train, scorez sur test.
3. Entraînez un `KNeighborsClassifier(n_neighbors=5)` sur les mêmes features L2-normalisées, scorez sur test.
4. Affichez les deux accuracies.

<details>
<summary>💡 Coup de pouce — pourquoi L2-normaliser, comparer logistic vs k-NN</summary>

**🎯 But :** comparer deux classifieurs légers sur les **mêmes** features pour mesurer le pouvoir discriminant intrinsèque de l'espace ViT.

**Pourquoi L2-normaliser ?**

Les features ViT peuvent avoir des magnitudes variables. La normalisation L2 ramène tous les vecteurs sur la sphère unité, ce qui :
- Rend la distance euclidienne entre 2 vecteurs équivalente à `2(1 - cos)` → on travaille en **similarité cosinus**.
- Stabilise la `LogisticRegression` (la régularisation L2 par défaut interagit moins mal).

```python
def l2(x):
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-8)

ftr = l2(feats_tr)
fte = l2(feats_te)
```

**Linear probe**

```python
clf = LogisticRegression(max_iter=1000, n_jobs=-1)
clf.fit(ftr, y_train)
acc_lr = clf.score(fte, y_test)
print(f"Linear probe : {acc_lr:.4f}")
```

⚠️ Sur 800 images train et 400 test, vous devriez atteindre **0.85 - 0.92** avec un ViT supervisé ImageNet. C'est très au-dessus de ce qu'on atteint avec HOG + SVM (TP du jour 3) qui plafonne autour de 0.5 - 0.6 sur STL-10.

**k-NN**

```python
knn = KNeighborsClassifier(n_neighbors=5, metric="euclidean", n_jobs=-1)
knn.fit(ftr, y_train)
acc_knn = knn.score(fte, y_test)
print(f"k-NN (k=5)   : {acc_knn:.4f}")
```

⚠️ Sur des features de qualité, **k-NN peut rivaliser avec le linear probe** sans aucun entraînement. C'est le signe que les classes sont bien séparées spatialement dans l'espace ViT.

**Ce que ces deux chiffres vous disent**
- Si linear probe et k-NN sont proches → les features sont vraiment discriminantes en soi.
- Si linear probe >> k-NN → les features sont linéairement séparables mais bruitées localement.
- Si les deux sont faibles → le backbone n'est pas adapté à la tâche (rare avec un ViT ImageNet sur des classes naturelles).

</details>


In [ ]:
# TODO

## Exercice 4 — Visualiser : t-SNE de l'espace ViT

Projetez les features test en 2D via t-SNE et coloriez par classe.
Si l'espace est bien séparable, on doit voir **10 clusters** distincts (un par classe).

1. L2-normalisez les features test.
2. Lancez `TSNE(n_components=2, perplexity=30, init="pca", random_state=0)`.
3. Tracez les points en colorant par classe, avec une légende.

<details>
<summary>💡 Coup de pouce — t-SNE sur les features test</summary>

**🎯 But :** voir directement si les classes forment des clusters dans l'espace ViT, indépendamment de tout classifieur.

**Calcul + plot**

```python
def plot_tsne(feats, y, class_names):
    feats_n = feats / (np.linalg.norm(feats, axis=1, keepdims=True) + 1e-8)
    emb = TSNE(
        n_components=2,
        perplexity=30,
        init="pca",
        random_state=0,
    ).fit_transform(feats_n)

    fig, ax = plt.subplots(figsize=(8, 7))
    for c in range(len(class_names)):
        m = y == c
        ax.scatter(emb[m, 0], emb[m, 1], s=14, alpha=0.7, label=class_names[c])
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
    ax.set_title("t-SNE des features ViT (test)"); ax.axis("off")
    plt.tight_layout(); plt.show()

plot_tsne(feats_te, y_test, class_names)
```

⚠️ **Astuce t-SNE** :
- `perplexity=30` est un bon défaut pour quelques centaines de points.
- `init="pca"` rend le résultat plus stable d'un run à l'autre que `init="random"`.
- t-SNE n'est **pas** un classifieur — c'est une visualisation. Les positions absolues ne veulent rien dire ; seules les **structures de voisinage** sont significatives.

**Ce que vous devriez observer**

Avec des features ViT supervisé ImageNet :
- Des **clusters bien séparés** pour les classes très distinctes (avion, voiture, bateau).
- Des **chevauchements** entre classes proches (chat / chien, cerf / cheval). Ce sont les confusions naturelles du modèle ImageNet.

C'est exactement le profil d'un espace de features où un linear probe atteint ~0.9 : globalement bien structuré, avec quelques recoupements là où la sémantique se chevauche.

</details>


In [ ]:
# TODO

## Pour aller plus loin

- **DINO / DINOv2** : remplacez le backbone par `vit_small_patch16_224.dino` (timm) ou `facebook/dinov2-small` (transformers). Ce sont des ViT pré-entraînés **sans labels** (self-supervised). Sur des tâches downstream comme STL-10, ils atteignent souvent des scores **supérieurs** au ViT supervisé.

- **Comparaison avec HOG + SVM (TP du jour 3)** : reprenez le pipeline HOG (`skimage.feature.hog`) sur ces mêmes images, entraînez un SVM ou une logistic. Le gain de précision avec ViT vs HOG illustre tout le saut entre représentations « hand-crafted » et représentations apprises à grande échelle.

- **CLIP zero-shot** : avec un modèle CLIP, on peut classifier STL-10 **sans aucun exemple d'entraînement**, en comparant chaque image au texte « a photo of a {class_name} ». Encore une marche au-dessus du paradigme « backbone gelé + tête entraînée ».
